In [5]:
import os 
os.chdir("/teamspace/studios/this_studio/mapping_reform")

import sys
sys.path.append(os.path.abspath("./slimer"))

import json


In [7]:
from vllm import LLM, SamplingParams
from src.SFT_finetuning.commons.prompter import SLIMER_instruction_prompter, Prompter

In [8]:
vllm_model = LLM("expertai/SLIMER", max_model_len=512, device="cpu")
# it is recommended to use a temperature of 0
# max_new_tokens can be adjusted depending on the expected length and number of entities (default 128)
sampling_params = SamplingParams(temperature=0, max_tokens=128, stop=['</s>'])

# , truncate_prompt_tokens=1)

INFO 07-16 16:02:35 [config.py:841] This model supports multiple tasks: {'generate', 'embed', 'reward', 'classify'}. Defaulting to 'generate'.
WARNING 07-16 16:02:35 [config.py:3320] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 07-16 16:02:35 [config.py:3371] Casting torch.bfloat16 to torch.float16.
INFO 07-16 16:02:35 [config.py:1472] Using max model len 512
WARNING 07-16 16:02:35 [arg_utils.py:1735] Compute Capability < 8.0 is not supported by the V1 Engine. Falling back to V0. 
INFO 07-16 16:02:36 [llm_engine.py:230] Initializing a V0 LLM engine (v0.9.2) with config: model='expertai/SLIMER', speculative_config=None, tokenizer='expertai/SLIMER', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=512, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 07-16 16:02:54 [default_loader.py:272] Loading weights took 17.24 seconds
INFO 07-16 16:02:55 [model_runner.py:1203] Model loading took 12.5524 GiB and 17.532936 seconds
INFO 07-16 16:02:57 [worker.py:294] Memory profiling takes 1.87 seconds
INFO 07-16 16:02:57 [worker.py:294] the current vLLM instance can use total_gpu_memory (14.75GiB) x gpu_memory_utilization (0.90) = 13.27GiB
INFO 07-16 16:02:57 [worker.py:294] model weights take 12.55GiB; non_torch_memory takes 0.03GiB; PyTorch activation peak memory takes 0.31GiB; the rest of the memory reserved for KV Cache is 0.38GiB.
INFO 07-16 16:02:57 [executor_base.py:113] # cuda blocks: 48, # CPU blocks: 512
INFO 07-16 16:02:57 [executor_base.py:118] Maximum concurrency for 512 tokens per request: 1.50x
INFO 07-16 16:02:59 [model_runner.py:1513] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 07-16 16:03:39 [model_runner.py:1671] Graph capturing finished in 40 secs, took 0.86 GiB
INFO 07-16 16:03:39 [llm_engine.py:428] init engine (profile, create kv cache, warmup model) took 44.13 seconds


In [9]:
entity_definitions = {
    "event": {
        "definition": "EVENT entities refer to specific manifestations of collective action for sociopolitical movements, such as meetings, protests, demonstrations, lectures, and gatherings.",
        "guidelines": "Avoid labeling events that are speculative, fictional, or allegorical. Label only events that have occurred and are being factually recounted."
    },
    "event_from_verb": {
        "definition": "EVENT_FROM_VERB are activities generated with reference to action verbs.",
        "guidelines": "Extract the activity from verbs, such as 'meeting' from 'met', 'lecture' from 'lectured', and 'march' from 'marched'."
    },
    "location": {
        "definition": "LOCATION entities refer to geographic places and proper nouns of specific locations, such as cities, counties, districts, and named buildings or places.",
        "guidelines": "Focus on geographic and administrative divisions, proper place names, and specific addresses. Extract as much locational information as is available. Example: extract all of 'Ship Inn, Long Lane, Bermondsey' rather than just 'Bermondsey'. Avoid labeling building types or generic spatial categories."
    },
    "space": {
        "definition": "SPACE entities describe types of physical spaces, buildings, or venues where people gather or activities take place.",
        "guidelines": "Look for words like: meeting room, school, factory, hall, building, house, shop, office, church, town, square, etc. Extract these words even when they appear in proper names like 'People's School' (extract 'school') or 'Market Square' (extract 'square'). Focus on the common noun that describes the type of space." 
        }
}


In [10]:
slimer_prompter = SLIMER_instruction_prompter("SLIMER_instruction_template", template_path='./slimer/src/SFT_finetuning/templates')
llama2_prompter = Prompter('LLaMA2-chat', template_path='./slimer/src/SFT_finetuning/templates', eos_text='')

def extract_entities(input_text):
    """Extract entities with streamlined output"""
    
    print(f"INPUT TEXT: \"{input_text}\"")
    print("\nENTITIES:")
    
    all_outputs = {}
    
    for tag, dng in entity_definitions.items():
        instruction = slimer_prompter.generate_prompt(
            ne_tag=tag,
            definition=dng["definition"],
            guidelines=dng["guidelines"]
        )
        prompt = llama2_prompter.generate_prompt(instruction, input_text)
        
        # Generate prediction (suppress the progress bars if possible)
        responses = vllm_model.generate([prompt], sampling_params)
        pred_response = responses[0].outputs[0].text.strip()
        
        # Parse the JSON output
        try:
            entities = json.loads(pred_response) if pred_response else []
        except:
            entities = []
        
        all_outputs[tag] = entities
        
        # Format output
        if entities:
            entities_str = ", ".join([f'"{entity}"' for entity in entities])
            print(f"{tag.upper()}: [{entities_str}]")
        else:
            print(f"{tag.upper()}: []")
    
    return all_outputs

Using prompt template SLIMER_instruction_template: SLIMER instruction template w or w/o D&G



In [1]:
input_text = "STALYBRIDGE.—A public meeting was held in the People’s School here on Monday evening last, when the National Petition was read and adopted; after which, Mr. James Leach, of Manchester, delivered an address, exposing the fallacies of the Corn Law repealers. A Corn Law lecture had been previously delivered in the town, by a Mr. Spencer, to about half a dozen of the middle classes; the Chartists, however, upset his meeting."
extract_entities(input_text)

NameError: name 'extract_entities' is not defined

In [11]:
simple_test = """
Weavers and card-room hands, attend the meeting which will be held in the Charlestown meeting room, on Wednesday evening, Nov. 6th, at eight o'clock, and show by your thousands that you are determined to be no longer 'stumped upon with impunity.
"""

extract_entities(simple_test)

INPUT TEXT: "
Weavers and card-room hands, attend the meeting which will be held in the Charlestown meeting room, on Wednesday evening, Nov. 6th, at eight o'clock, and show by your thousands that you are determined to be no longer 'stumped upon with impunity.
"

ENTITIES:


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

EVENT: ["meeting"]


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

LOCATION: ["Charlestown"]


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

SPACE: ["Charlestown meeting room"]


{'event': ['meeting'],
 'event_from_verb': [],
 'location': ['Charlestown'],
 'space': ['Charlestown meeting room']}

In [20]:
test_text = """
A comfortable supper party met at the Chequers Inn in the evening, but Mr. O'Connor could not be present. All went off with harmony and glee. This meeting has brought upwards of £20 to the Char- ity funds.  DUBLIN.—The Irish Universal Suffrage Associa- tion met on Sunday last, at their great room, 14, North Anne-street, Mr. P. O'Connell in the chair. The Secretary read the minutes of the last meeting; he also read letters from the following persons—Mr. Thomas Cooper, Leicester; Mr. E. Mayne, Wakefield; Mr. K. Macroy, Aberdeen; Mr. John Baldwin, London; Mr. Daniel M'Intosh, Glasgow and Mr. William Campbell, Manchester; all giving abundant proof that the people of England and Scot- land seek nothing for themselves that they do not wish the people of Ireland to be equal participators in. The Secretary also moved that Mr. John Little, Mr. G. Watkins, and Mr. John Matson be admitted members; after which, Mr. O'Higgins rose and brought forward his promised motion relative to Mr. Sharman Crawford's Landlord and Tenant Bill. Mr. O'Higgins made a long and excellent speech, which we received only a few hours before going to press, and which we have no room for. He con- cluded by moving the following resolution:—"That it is contrary to every principle of natural justice, as well as a direct violation of the laws of God, to deprive any man of the fruits of his labour, without remuneration; and inasmuch as it is the common and uniform practice of the majority of Irish landlords to turn out great numbers of their tenants, under the pretence of clearing their estates of a 'superabundant population,'
"""
extract_entities(test_text)

INPUT TEXT: "
A comfortable supper party met at the Chequers Inn in the evening, but Mr. O'Connor could not be present. All went off with harmony and glee. This meeting has brought upwards of £20 to the Char- ity funds.  DUBLIN.—The Irish Universal Suffrage Associa- tion met on Sunday last, at their great room, 14, North Anne-street, Mr. P. O'Connell in the chair. The Secretary read the minutes of the last meeting; he also read letters from the following persons—Mr. Thomas Cooper, Leicester; Mr. E. Mayne, Wakefield; Mr. K. Macroy, Aberdeen; Mr. John Baldwin, London; Mr. Daniel M'Intosh, Glasgow and Mr. William Campbell, Manchester; all giving abundant proof that the people of England and Scot- land seek nothing for themselves that they do not wish the people of Ireland to be equal participators in. The Secretary also moved that Mr. John Little, Mr. G. Watkins, and Mr. John Matson be admitted members; after which, Mr. O'Higgins rose and brought forward his promised motion relative to Mr

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

ValueError: The decoder prompt (length 650) is longer than the maximum model length of 512. Make sure that `max_model_len` is no smaller than the number of text tokens.

In [12]:
test_short = """
DUBLIN.—The Irish Universal Suffrage Associa- tion met on Sunday last, at their great room, 14, North Anne-street, Mr. P. O'Connell in the chair. The Secretary read the minutes of the last meeting; he also read letters from the following persons—Mr. Thomas Cooper, Leicester; Mr. E. Mayne, Wakefield; Mr. K. Macroy, Aberdeen; Mr. John Baldwin, London; Mr. Daniel M'Intosh, Glasgow and Mr. William Campbell, Manchester; all giving abundant proof that the people of England and Scot- land seek nothing for themselves that they do not wish the people of Ireland to be equal participators in. 
"""
extract_entities(test_short)

INPUT TEXT: "
DUBLIN.—The Irish Universal Suffrage Associa- tion met on Sunday last, at their great room, 14, North Anne-street, Mr. P. O'Connell in the chair. The Secretary read the minutes of the last meeting; he also read letters from the following persons—Mr. Thomas Cooper, Leicester; Mr. E. Mayne, Wakefield; Mr. K. Macroy, Aberdeen; Mr. John Baldwin, London; Mr. Daniel M'Intosh, Glasgow and Mr. William Campbell, Manchester; all giving abundant proof that the people of England and Scot- land seek nothing for themselves that they do not wish the people of Ireland to be equal participators in. 
"

ENTITIES:


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

EVENT: []


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

...

LOCATION: ["14, North Anne-street"]


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

SPACE: []


{'event': [],
 'event_from_verb': [],
 'location': ['14, North Anne-street'],
 'space': []}